# Missing Values

Missing data is a fundamental problem in real-world data science. Pandas represents missing data primarily with the Numpy `NaN` (Not a Number) object.

We will explore:
- Finding missing values with `isna()`
- Discarding missing values with `dropna()`
- Injecting default values with `fillna()`
- Time-sensitive data filling (`ffill` & `bfill`)
- Replacing incorrect values with `replace()`

In [1]:
# Import pandas
import pandas as pd

## Identifying and Cleaning Missing Data

We load a dataset of student grades that contains missing assignments.

In [2]:
# Load class_grades dataset
df = pd.read_csv("datasets/class_grades.csv")
df.head()

,Prefix,Assignment,Tutorial,Midterm,TakeHome,Final
0,5,57.14,34.09,64.38,51.48,52.50
1,8,95.05,105.49,67.50,99.07,68.33
2,8,83.70,83.17,NaN,63.15,48.89
3,7,NaN,NaN,49.38,105.93,80.56
4,8,91.32,93.64,95.00,107.41,73.89


In [3]:
# Generate a DataFrame of booleans indicating where missing data occurs
masks = df.isna()
masks.head()

,Prefix,Assignment,Tutorial,Midterm,TakeHome,Final
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,True,False,False
3,False,True,True,False,False,False
4,False,False,False,False,False,False


In [4]:
# Drop any row containing at least one NaN
df.dropna().head()

,Prefix,Assignment,Tutorial,Midterm,TakeHome,Final
0,5,57.14,34.09,64.38,51.48,52.50
1,8,95.05,105.49,67.50,99.07,68.33
4,8,91.32,93.64,95.00,107.41,73.89
5,7,95.00,92.58,93.12,97.78,68.06
6,8,95.05,102.99,56.25,99.07,50.00


In [5]:
# Instead of dropping, we can fill missing data. Here we fill NaNs with 0.
df.fillna(0, inplace=True)
df.head()

,Prefix,Assignment,Tutorial,Midterm,TakeHome,Final
0,5,57.14,34.09,64.38,51.48,52.50
1,8,95.05,105.49,67.50,99.07,68.33
2,8,83.70,83.17,0.00,63.15,48.89
3,7,0.00,0.00,49.38,105.93,80.56
4,8,91.32,93.64,95.00,107.41,73.89


## Context-Aware Filling for Time Series Data

When tracking event logs or progressive states, dropping or zeroing out intermediate missing statuses isn't logical.

Instead, we can propagate the last valid observation forward until the next valid hit via **Forward Fill (`ffill`)**.

In [6]:
# We load server logs containing playback tracking metrics
df = pd.read_csv("datasets/log.csv")
df.head()

,time,user,video,playback position,paused,volume
0,1469974424,cheryl,intro.html,5,False,10.0
1,1469974454,cheryl,intro.html,6,NaN,NaN
2,1469974544,cheryl,intro.html,9,NaN,NaN
3,1469974574,cheryl,intro.html,10,NaN,NaN
4,1469977514,bob,intro.html,1,NaN,NaN


In [7]:
# It's crucial to sort the data chronically before doing fill propagation
df = df.set_index("time").sort_index()
df.head()

,user,video,playback position,paused,volume
time,,,,,
1469974424,cheryl,intro.html,5,False,10.0
1469974424,sue,advanced.html,23,False,10.0
1469974454,sue,advanced.html,24,NaN,NaN
1469974454,cheryl,intro.html,6,NaN,NaN
1469974484,cheryl,intro.html,7,NaN,NaN


If different users have independent event timelines entangled together, mixing their states ruins everything. 
We can structure a `MultiIndex` grouping time events by the `user` before calling `.ffill()`.

In [8]:
# Set hierarchical index layout: Time -> User
df = df.reset_index()
df = df.set_index(["time", "user"])
df.head()

video  playback position paused  volume
time       user                                                   
1469974424 cheryl     intro.html                  5  False    10.0
           sue     advanced.html                 23  False    10.0
1469974454 sue     advanced.html                 24    NaN     NaN
           cheryl     intro.html                  6    NaN     NaN
1469974484 cheryl     intro.html                  7    NaN     NaN

In [9]:
# Forward fill. Notice how missing 'paused' or 'volume' metrics adopt the preceding status value
df = df.ffill()
df.head()

C:\Users\kanko\AppData\Local\Temp\ipykernel_14152\3191061569.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.ffill()


video  playback position  paused  volume
time       user                                                    
1469974424 cheryl     intro.html                  5   False    10.0
           sue     advanced.html                 23   False    10.0
1469974454 sue     advanced.html                 24   False    10.0
           cheryl     intro.html                  6   False    10.0
1469974484 cheryl     intro.html                  7   False    10.0

## Data Replacement Patterns

Sometimes data is present, but fundamentally erroneous or formatted improperly. You can sanitize this using the `.replace()` module.

In [10]:
# Make a mock DataFrame
df = pd.DataFrame({'A': [1, 1, 2, 3, 4],
                   'B': [3, 6, 3, 8, 9],
                   'C': ['a', 'b', 'c', 'd', 'e']})
df

,A,B,C
0,1,3,a
1,1,6,b
2,2,3,c
3,3,8,d
4,4,9,e


In [11]:
# Replace all global occurrences of the exact value '1' with '100'
df.replace(1, 100)

,A,B,C
0,100,3,a
1,100,6,b
2,2,3,c
3,3,8,d
4,4,9,e


### Regex-Powered Replacements

You can deploy powerful Regular Expressions natively within Pandas `.replace()` calls to intercept dynamic patterns.

In [12]:
# Reload uncleaned original logs
df = pd.read_csv("datasets/log.csv")

# Substitute any string terminating in '.html' with 'webpage'
df.replace(to_replace=r".*.html$", value="webpage", regex=True).head()

,time,user,video,playback position,paused,volume
0,1469974424,cheryl,webpage,5,False,10.0
1,1469974454,cheryl,webpage,6,NaN,NaN
2,1469974544,cheryl,webpage,9,NaN,NaN
3,1469974574,cheryl,webpage,10,NaN,NaN
4,1469977514,bob,webpage,1,NaN,NaN
